In [34]:
import re
import os
import json
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import AutoTokenizer, AutoModelForCausalLM
from itertools import compress

is_cuda = torch.cuda.is_available()
num = torch.cuda.device_count()
name = torch.cuda.get_device_name(0) if is_cuda and num > 0 else "None"

device = "cuda"

hub = Path(os.path.expanduser("~/.cache/huggingface/hub"))
base = hub / "models--codellama--CodeLlama-7b-Instruct-hf"

# Prefer the ref in refs/main; fall back to the newest snapshot
ref_file = base / "refs" / "main"
commit = ref_file.read_text().strip()

MODEL_PATH = str(base / "snapshots" / commit)
print("MODEL_PATH =", MODEL_PATH)

MODEL_PATH = /home/rpinter/.cache/huggingface/hub/models--codellama--CodeLlama-7b-Instruct-hf/snapshots/22cb240e0292b0b5ab4c17ccd97aa3a2f799cbed


In [8]:
# tokenizer
tok = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    use_fast=True,
    local_files_only=True,
)
from transformers import BitsAndBytesConfig

q_config = BitsAndBytesConfig(
   load_in_4bit=True,
   bnb_4bit_quant_type="nf4",
   bnb_4bit_use_double_quant=True,
   bnb_4bit_compute_dtype=torch.bfloat16
) 

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
    device_map="auto",
    low_cpu_mem_usage=True,
    quantization_config=q_config,
    attn_implementation="sdpa", # flash attention
    dtype=torch.bfloat16, # and changing dtype to lower precision
)

def ask_q(prompt, max_new_tokens=128):
    inst = f"<s>[INST] {prompt.strip()} [/INST]"
    inputs = tok(inst, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            pad_token_id=tok.eos_token_id,
            use_cache=True,
        )
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [44]:
def read_file_to_string(path):
    with open(path) as f:
        return f.read()

def get_json_template(n):
    return json.dumps(
        {f'{k}': "<true/false>" for k in range(1,n+1)},
        indent=2,
        ensure_ascii=False)

def extract_json_object(text):
    m = re.search(r'\{.*?\}', text, flags=re.DOTALL)
    return m.group(0) if m else None


def to_bool(v):
    return v if isinstance(v, bool) else str(v).strip().lower() == "true"


In [15]:
examples_path = "../examples/"
src_files = [
    src_file 
    for src_file in os.listdir(examples_path)
        if ("json" not in src_file) and (os.path.isfile(examples_path + src_file))
    ]
src_files

['interactive_bezier.js',
 'sq-circles.js',
 'flow_field_with_letters.js',
 'plein014-core.js',
 'sound003.html',
 'plottable004.html',
 'bingo_card.js',
 'black&white_rotate.js',
 'vertical_BW_stripes.js',
 'waves.js',
 'lerp-noise.js',
 'gaspe004.js',
 'flowFieldNoiseAudio.js']

In [27]:
material_process = {
    "audio file": "1) Does it use external audio file? [True/False]",
    "image file": "2) Does it use external image file? [True/False]",
    "sound": "3) Does it generate sound? [True/False]",
    "image": "4) Does it generate images? [True/False]",
    "randomness": "5) Does it contain randomness? [True/False]",
    "interaction": "6) Does it contain interactions? [True/False]",
}
# env_interactions = [
#     "1) Does it depend on human interaction (through mouse, midi controller, microphone, keyboard, camera, motion sensor, or lidar) to run? [True/False]",
#     "2) Does it depend on computer interaction (specifically through data file, data stream, remote data, or web API) to run? [True/False]",
# ]

env_interactions_2 = {
    "human": "1) Does it depend on human interaction (through mouse, midi controller, microphone, keyboard, camera, motion sensor, or lidar) to run? [True/False]",
    "data file": "2) Does it depend on computer interaction through data files to run? [True/False]",
    "data stream": "3) Does it depend on computer interaction through data stream to run? [True/False]",
    "remote data": "4) Does it depend on computer interaction through remote data to run? [True/False]",
    "web API": "5) Does it depend on computer interaction through web API to run? [True/False]"
 }

sensory_outcomes = {
    "visual": "1) Does it produce visual sensory outcomes? [True/False]",
    "auditory": "2) Does it produce auditory sensory outcomes? [True/False]",
    "physical": "3) Does it produce physical sensory outcomes? [True/False]",
    "static": "4) Does it produce static sensory outcomes? [True/False]",
    "time-based": "5) Does it produce time-based sensory outcomes? [True/False]"
}

In [49]:
for f in src_files:
    answers = {}
    for tag, questions in zip(
        ["entities", "interactions", "outcomes"], 
        [material_process, env_interactions_2, sensory_outcomes]
    ):
        code_str = read_file_to_string(examples_path + f)
        q_list = questions.keys()
        prompt = """
        You are a helpful assistent. You must only return valid json files and no other text.
        Considering this code:
        ```
        {code_str}
        ```
        Create a JSON list with boolean values with the answers for the following questions:
        
        {question}
        
        Now write the valid enumerated json list file with answers and nothing else using the following template:
        
        {json_template}
    
        Remember to return ONLY JSON.
        """.format(
            code_str=code_str, 
            question=q_list, 
            json_template=get_json_template(len(q_list)))
        
        response = ask_q(prompt)
        # extract json with regex
        response = json.loads(extract_json_object(response))
        bool_responses = [to_bool(response.get(str(i), False)) for i in range(1, len(q_list)+1)]

        answers[tag] = list(compress(q_list, bool_responses))
        base, _ext = os.path.splitext(f)
        out_path = os.path.join(examples_path + "results", f"{base}.json")
        with open(out_path, "w", encoding="utf-8") as fp:
            json.dump(answers, fp, ensure_ascii=False, indent=2)
        print(f"Saved: {out_path}")

{'1': True, '2': False, '3': False, '4': False, '5': False, '6': False}
Saved: ../examples/results/interactive_bezier.json
{'1': 'true', '2': 'false', '3': 'false', '4': 'false', '5': 'false'}
Saved: ../examples/results/interactive_bezier.json
{'1': False, '2': False, '3': False, '4': False, '5': False}
Saved: ../examples/results/interactive_bezier.json
{'1': 'true', '2': 'false', '3': 'false', '4': 'false', '5': 'false', '6': 'false'}
Saved: ../examples/results/sq-circles.json
{'1': 'true', '2': 'false', '3': 'false', '4': 'false', '5': 'false'}
Saved: ../examples/results/sq-circles.json
{'1': False, '2': False, '3': False, '4': False, '5': False}
Saved: ../examples/results/sq-circles.json
{'1': True, '2': False, '3': False, '4': False, '5': False, '6': False}
Saved: ../examples/results/flow_field_with_letters.json
{'1': True, '2': False, '3': False, '4': False, '5': False}
Saved: ../examples/results/flow_field_with_letters.json
{'1': False, '2': False, '3': False, '4': False, '5': Fa